In [7]:
from functools import lru_cache
from typing import Dict, Any
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
# import re
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer


%matplotlib inline

# Dataframe loading

In [17]:
df_prod = pd.read_csv('data/20250501-carrefour_prods.csv', parse_dates=['dateKey'])
df_loyalty = pd.read_csv('data/20250501-carrefour_loyalty.csv', parse_dates=['date'])
df_prod['totalPriceAfterDiscount'] = df_prod['totalPrice'] + df_prod['totalImmediateDiscount']
display(df_prod)
display(df_loyalty)

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalPriceAfterDiscount
0,2025-04-29,receipt,2,NaN,NaN,5X10 D FREEDENT ME,5X10 D FREEDENT ME,other,NaN,20.0,2,0.000,2.39,4.78,-1.43,3.35
1,2025-04-29,receipt,1,NaN,NaN,KG POULET FERMIER,KG POULET FERMIER,food,NaN,5.5,1,1.912,6.50,12.43,0.00,12.43
2,2025-04-29,receipt,1,NaN,NaN,750G OMEGA 3 DX SH,750G OMEGA 3 DX SH,other,NaN,20.0,1,0.000,5.39,5.39,0.00,5.39
3,2025-04-29,receipt,1,NaN,NaN,3 RECH RUBAN MAGI,3 RECH RUBAN MAGI,other,NaN,20.0,1,0.000,4.99,4.99,0.00,4.99
4,2025-04-29,receipt,1,NaN,NaN,132G BISC.COCO SSA,132G BISC.COCO SSA,food,NaN,5.5,1,0.000,2.45,2.45,0.00,2.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5014,2022-04-25,receipt,1,NaN,NaN,COURGETTE P10.5 CM,COURGETTE P10.5 CM,other,NaN,10.0,1,0.000,1.75,1.75,0.00,1.75
5015,2022-04-25,receipt,1,NaN,NaN,POMME ARIANE,POMME ARIANE,food,NaN,5.5,1,0.000,2.50,2.50,0.00,2.50
5016,2022-04-25,receipt,1,NaN,NaN,PIECE ANANAS EXTRA,PIECE ANANAS EXTRA,food,NaN,5.5,1,0.000,1.99,1.99,0.00,1.99
5017,2022-04-25,receipt,1,NaN,NaN,MENTHE FRAI,MENTHE FRAI,food,NaN,5.5,1,0.000,0.61,0.61,0.00,0.61


,operationId,date,earned,burned,itemLabel,promotionLabel,itemRd,loyaltyOperation
0,64070450536,2024-09-28,0.66,0.00,20 OEUFS DJP POULE AU SOL CRF,NaN,0.46,Paiement en caisse
1,64070450536,2024-09-28,0.66,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.20,Paiement en caisse
2,64063052302,2024-09-24,0.30,0.00,1KG COUSCOUS MOYEN CRF,NaN,0.30,Paiement en caisse
3,64072667140,2024-09-20,0.20,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.20,Paiement en ligne
4,64073135027,2024-09-13,0.00,-22.76,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
329,65151220627,2025-03-08,21.30,0.00,"ART RAYON FRUITS LEG TVA 5,5",NaN,1.29,Paiement en caisse
330,65151220627,2025-03-08,21.30,0.00,"ART RAYON POISSONNERIE TVA 5,5",NaN,12.91,Paiement en caisse
331,65151220627,2025-03-08,21.30,0.00,CHOU FLEUR PIECE,NaN,0.26,Paiement en caisse
332,65151220627,2025-03-08,21.30,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.25,Paiement en caisse


In [18]:
mapped_categories = {
    'Charcuterie' : 'food', 
    'Laits et Boissons végétales' : 'food',
    'Jus de fruits et légumes': 'food', 
    'Toasts et Pains de mie': 'food',
    'Yaourts et Fromages blancs': 'food',
    'Conserves et Bocaux': 'food',
    'Colas, Thés glacés, Sirops et Sodas': 'food',
    'Légumes': 'food',
    'Huiles, Vinaigres et Vinaigrettes': 'food',
    'Nettoyants vaisselle' : 'other',
    'Accessoires de ménage' : 'other',
    'Matériel de bureau' : 'other',
    'Fromages': 'food',
    'Epicerie salée': 'food',
    'Cheveux' : 'other',
    'Pains Burger, Sandwich et Wraps': 'food',
    'Eaux': 'food',
    'Viandes': 'food',
    'Lessives' : 'other',
    'Pizzas, Quiches et Tartes': 'food',
    'Apéritifs et Chips': 'food',
    'Fruits': 'food', 
    'Volaille et Rôtisserie': 'food',
    'Glaces et Sorbets': 'food',
    'Bio à Petit prix': 'food',
    'Gâteaux moelleux': 'food',
    'Apéritifs, Entrées et Snacking': 'food', 
    'Petit déjeuner': 'food',
    'Hygiène dentaire' : 'other', 
    'Cave à Vins': 'food',
    'Boucherie': 'food',
    'Poissons et Fruits de mer': 'food',
    'Œufs': 'food',
    'Poissonnerie': 'food',
    'Essuie-tout, Papier toilette et Mouchoirs' : 'other',
    'Confiseries et Chocolats': 'food', 
    'RETURNABLE_BAG' : 'other', 
    'Hygiène intime ' : 'other',
    'Désodorisants et Bougies' : 'other',
    'Toutes nos régions': 'food',
    'Produits nettoyants' : 'other', 
    'Riz, Purées et Féculents' : 'food',
    'Ingrédients pour cuisiner' : 'food',
    'Sauces froides' : 'food',
    'Pains frais' : 'food',
    'Bières et Cidres' : 'food',
    'Repas de Pâques' : 'food',
    'Le Marché' : 'food',
    'Beurres et Crèmes' : 'food',
    'Premiers soins et Préservatifs' : 'other',
    'Nintendo Switch' : 'other',
    'Sucres, Farines et Aide à la pâtisserie' : 'food',
    'Viennoiseries et Brioches fraîches' : 'food', 
    'Corps' : 'other'
}

In [19]:
df_prod.loc[df_prod.category.isna(), "category"] = df_prod.loc[df_prod.category.isna(), "subCategory"].map(mapped_categories)

In [20]:
def input_from_csv(
    df: pd.DataFrame,
    filepath: str,
    column_name: str,
):
    """In-place computation of embedding"""
    input_df = pd.read_csv(filepath)
    if isinstance(column_name, str):
        merged_df = pd.merge(df, input_df, on=column_name, how="left", suffixes=("_original", "_imputed"))
    else:
        raise ValueError("column_name should be a string")
    return merged_df

In [21]:
df_prod = input_from_csv(
    df_prod,
    filepath='data/20250502-carrefour_food_products_labels_most_2.csv',
    column_name='productLabel'
)

In [22]:
df_prod.query('productLabel.notna()')

,dateKey,recordType,countVisits,ean,cdbase,productLabel,slugProductLabel,category,subCategory_original,vatPercentage,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalPriceAfterDiscount,subCategory_imputed
0,2025-04-29,receipt,2,NaN,NaN,5X10 D FREEDENT ME,5X10 D FREEDENT ME,other,NaN,20.0,2,0.000,2.39,4.78,-1.43,3.35,NaN
1,2025-04-29,receipt,1,NaN,NaN,KG POULET FERMIER,KG POULET FERMIER,food,NaN,5.5,1,1.912,6.50,12.43,0.00,12.43,Volaille et Rôtisserie
2,2025-04-29,receipt,1,NaN,NaN,750G OMEGA 3 DX SH,750G OMEGA 3 DX SH,other,NaN,20.0,1,0.000,5.39,5.39,0.00,5.39,NaN
3,2025-04-29,receipt,1,NaN,NaN,3 RECH RUBAN MAGI,3 RECH RUBAN MAGI,other,NaN,20.0,1,0.000,4.99,4.99,0.00,4.99,NaN
4,2025-04-29,receipt,1,NaN,NaN,132G BISC.COCO SSA,132G BISC.COCO SSA,food,NaN,5.5,1,0.000,2.45,2.45,0.00,2.45,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5014,2022-04-25,receipt,1,NaN,NaN,COURGETTE P10.5 CM,COURGETTE P10.5 CM,other,NaN,10.0,1,0.000,1.75,1.75,0.00,1.75,NaN
5015,2022-04-25,receipt,1,NaN,NaN,POMME ARIANE,POMME ARIANE,food,NaN,5.5,1,0.000,2.50,2.50,0.00,2.50,NaN
5016,2022-04-25,receipt,1,NaN,NaN,PIECE ANANAS EXTRA,PIECE ANANAS EXTRA,food,NaN,5.5,1,0.000,1.99,1.99,0.00,1.99,Conserves et Bocaux
5017,2022-04-25,receipt,1,NaN,NaN,MENTHE FRAI,MENTHE FRAI,food,NaN,5.5,1,0.000,0.61,0.61,0.00,0.61,NaN


In [23]:
df_prod.to_csv('data/20250501-carrefour_products.csv', index=False)

# Group analysis

The method unstack allows to view the `groupby` results in a natural table. This is actually a pivot on the column by which one applies `groupby`. 
The `reset_index()` method actually returns it as a table with index as rows instead (no pivoting). One or another is useful.

In [9]:
df_loyalty[['itemLabel', 'date', 'loyaltyOperation', 'itemRd']].groupby(by=[df_loyalty.date.dt.year, 'itemLabel'], sort=True).sum(numeric_only=True).sort_values(by=['date', 'itemRd'], ascending = [False, False]).unstack(fill_value=0)

itemRd                                 \
itemLabel 100G GRANA PADANO RAPE AOP CRF 125G MOZZARELLA BUFALA BIO CRF   
date                                                                      
2024                                0.00                           0.00   
2025                                0.58                           0.77   

                                                             \
itemLabel 125G MOZZARELLA VACHE 12X125G YAOURTS PAT LA LAIT   
date                                                          
2024                       1.19                        0.00   
2025                       0.00                        1.34   

                                                                        \
itemLabel 16X125G BIF NATURE ACTIVIA OD 16X125G YRT FRUIT 0% CRF CLASS   
date                                                                     
2024                               1.35                           0.61   
2025                               0.00                           0.00   

                                                                 \
itemLabel 1KG COUSCOUS MOYEN CRF 1KG FARINE BLE FLUID T45 CRF C   
date                                                              
2024                         0.3                           0.13   
2025                         0.0                           0.00   

                                                               ...  \
itemLabel 1KG FARINE BLE T45 CRF CLASSIC 1KG FILET POULET PLK  ...   
date                                                           ...   
2024                                0.13                 0.00  ...   
2025                                0.00                 1.99  ...   

                                                           \
itemLabel RICORE RECHARGE 290G RUBAN TRANSPT 550 19MMX33M   
date                                                        
2024                      0.00                       1.93   
2025                      3.24                       0.00   

                                                       \
itemLabel SALADE BATAVIA PIECE ST 1,5KG POMME GALA FR   
date                                                    
2024                      0.37                   0.64   
2025                      0.00                   0.00   

                                                                               \
itemLabel ST 200G HARICOT.MUNGO 0,99 ST 2PCES MAIS DOUX HF ST 400G EPINARD FR   
date                                                                            
2024                            0.15                   0.0                0.0   
2025                            0.00                   0.9                0.5   

                                                                    \
itemLabel ST 500G BETTERAV PRIM FQC AGRO ST 500G BETTERAVE FQC AGR   
date                                                                 
2024                                0.27                      0.25   
2025                                0.00                      0.00   

                                          
itemLabel SWITCH SET MARIO KART 8 PASS C  
date                                      
2024                                10.5  
2025                                 0.0  

[2 rows x 102 columns]

In [ ]:
grouped_loyalty_year = df_loyalty[['itemLabel', 'date', 'loyaltyOperation', 'itemRd']].groupby(by=[df_loyalty.date.dt.year, 'itemLabel'], sort=True).sum(numeric_only=True).sort_values(by=['date', 'itemRd'], ascending = [False, False]).reset_index()
display(grouped_loyalty_year.head(10))

In [ ]:
display(grouped_loyalty_year[grouped_loyalty_year.date == 2024].nlargest(10, columns='itemRd'))

,date,itemLabel,itemRd
58,2024,"ART RAYON POISSONNERIE TVA 5,5",15.81
59,2024,SWITCH SET MARIO KART 8 PASS C,10.50
60,2024,"ART RAYON FRUITS LEG TVA 5,5",9.84
61,2024,BANANE CRF BIO MH 5 FRUITS,9.09
62,2024,2X6TR 2X215G SF NOR,6.60
63,2024,2X75ML DENT PSA NETT INTENS OB,4.70
64,2024,2X75ML DENT S&G CALM ORIGIN OB,4.25
65,2024,2X75ML PSA GENC&EMAIL ORIG OB,4.24
66,2024,1X8L FF BP MATIN LEGER ECREME,4.08
67,2024,3X75ML DENT WN TP SENS SIGNAL,4.07


In [103]:
# Group by 'category' and sum the 'amount' column
spending_by_category = df_prod.groupby(['category', df_prod.dateKey.dt.year]).sum(numeric_only=True)

print("\nTotal Spending by Category:")
display(spending_by_category)


Total Spending by Category:


countVisits           ean       cdbase  vatPercentage  \
category dateKey                                                          
food     2022             947  0.000000e+00          0.0         4482.5   
         2023            1620  0.000000e+00          0.0         7386.5   
         2024            1745  6.645180e+14  781106656.0         7293.0   
         2025             533  7.403638e+13  100131650.0         2370.5   
other    2022             340  0.000000e+00          0.0         4660.0   
         2023             476  0.000000e+00          0.0         6010.0   
         2024             465  3.411904e+14  313664675.0         4640.0   
         2025              88  6.492628e+13   55207784.0          960.0   

                  totalQuantity  totalWeight    unitPrice  totalPrice  \
category dateKey                                                        
food     2022              1163       91.723  2559.725000     3046.18   
         2023              1938      192.193  4475.635333     5454.36   
         2024              2064      239.980  4858.790000     5683.34   
         2025               614       98.519  1603.980000     1904.38   
other    2022               331       -5.721  1519.430000     1763.48   
         2023               474        0.000  2543.440000     3165.19   
         2024               551       -1.200  1937.386667     2816.82   
         2025               103        0.000   352.610000      609.50   

                  totalImmediateDiscount  totalTruePrice  
category dateKey                                          
food     2022                    -133.34         2912.84  
         2023                    -330.49         5123.87  
         2024                    -165.56         5517.78  
         2025                     -63.04         1841.34  
other    2022                    -165.00         1598.48  
         2023                    -382.10         2783.09  
         2024                    -272.27         2544.55  
         2025                     -41.68          567.82

In [ ]:
df_prod['yearMonth'] = df_prod.dateKey.dt.to_period('M')
# Group by year_month and product_id, then sum the quantities
grouped_month = df_prod.groupby(['yearMonth', 'productLabel']).agg(
    {'totalQuantity' : 'sum', 'totalWeight': 'sum', 'unitPrice' : 'mean', 'totalPrice' : 'sum', 'totalImmediateDiscount' : 'sum', 'totalTruePrice' : 'sum'}
).reset_index()

# Sort by year_month and quantity in descending order
sorted_grouped_month_quant = grouped_month.sort_values(by=['yearMonth', 'totalQuantity'], ascending=[True, False])
sorted_grouped_month_weight = grouped_month.sort_values(by=['yearMonth', 'totalWeight'], ascending=[True, False])

# Get the top 10 items for each month
top5_quant_by_month = sorted_grouped_month_quant.groupby('yearMonth').head(5)  # Top N records per group
top5_weight_by_month = sorted_grouped_month_weight.groupby('yearMonth').head(5)  # Top N records per group
display(top5_quant_by_month[top5_quant_by_month.yearMonth.dt.year == 2024])
display(top5_weight_by_month[top5_weight_by_month.yearMonth.dt.year == 2024])

,yearMonth,productLabel,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice
2444,2024-01,KIWI JAUNE PIECE,20,0.000,0.723333,14.20,-2.32,11.88
2446,2024-01,KIWI VERT PIECE,12,0.000,0.640000,7.48,-1.88,5.60
2382,2024-01,24 PS EXTRA LONG C,9,0.000,1.550000,13.95,-1.24,12.71
2405,2024-01,6X1.5L CRISTALINE,9,0.000,1.140000,10.26,0.00,10.26
2445,2024-01,KIWI VERT FQC,8,0.000,0.690000,5.52,-1.52,4.00
2582,2024-02,KIWI VERT PIECE,17,0.000,0.590000,10.03,-1.74,8.29
2601,2024-02,Nettoyant Optique Dégraissant Anti Trace VU,8,0.000,2.790000,22.32,0.00,22.32
2536,2024-02,6X1.5L CRISTALINE,7,0.000,1.140000,7.98,0.00,7.98
2597,2024-02,Mouchoirs Confort CARREFOUR SOFT,6,0.000,3.790000,22.74,3.42,26.16
2568,2024-02,CITRON VERT PIECE,5,0.000,0.500000,2.50,-0.50,2.00


,yearMonth,productLabel,totalQuantity,totalWeight,unitPrice,totalPrice,totalImmediateDiscount,totalTruePrice
2453,2024-01,PDT CONSOMMATION,5,4.487,1.898000,10.46,0.0,10.46
2477,2024-01,TOMATE RDE CHARNUE,2,2.010,2.490000,5.27,0.0,5.27
2467,2024-01,POMME PINK LADY KG,2,1.744,1.990000,3.47,0.0,3.47
2451,2024-01,PATATE DOUCE VRAC,1,1.724,1.990000,3.43,0.0,3.43
2466,2024-01,POMME ARIANE VRAC,1,1.304,2.990000,3.90,0.0,3.90
2611,2024-02,PDT CONSOMMATION,2,2.854,1.990000,5.68,0.0,5.68
2590,2024-02,MANDARINE CRF,2,1.959,1.990000,3.90,0.0,3.90
2626,2024-02,POMME GRANNY GROSS,2,1.572,2.990000,4.70,0.0,4.70
2562,2024-02,CAROTTE VRAC,1,1.036,2.490000,2.58,0.0,2.58
2627,2024-02,POMME PINK LADY KG,1,0.915,1.990000,1.82,0.0,1.82


In [ ]:
# Group by year_month and product_id, then sum the quantities
grouped_year = df_prod.groupby(['year', 'productLabel']).agg(
    {'totalQuantity' : 'sum', 'totalWeight': 'sum', 'unitPrice' : 'mean', 'price' : 'sum', 'totalImmediateDiscount' : 'sum'}).reset_index()

# Sort by year_month and quantity in descending order
sorted_grouped_year_quant = grouped_year.sort_values(by=['year', 'totalQuantity'], ascending=[True, False])
sorted_grouped_year_weight = grouped_year.sort_values(by=['year', 'totalWeight'], ascending=[True, False])

# Get the top 10 items for each month
top10_quant_by_year = sorted_grouped_year_quant.groupby('year').head(10)  # Top N records per group
top10_weight_by_year = sorted_grouped_year_weight.groupby('year').head(10)  # Top N records per group
display(top10_quant_by_year)
display(top10_weight_by_year)

,year,name,totalQuantity,totalWeight,unitPrice,price,immediateDiscount
447,2022,KIWI PIECE,129,0.000,0.564000,4.893333,-0.390000
51,2022,16X125 VELOUTE NAT,19,0.000,3.778750,3.778750,0.000000
356,2022,CHAUSSON AUX POMME,18,0.000,0.750000,1.934000,0.000000
291,2022,AVOCAT PIECE,16,0.000,1.170000,3.888000,-0.266000
372,2022,CONCOMBRE PIECE,16,0.000,1.111250,2.222500,-0.022500
614,2022,VIENNOISERIE AUX A,16,0.000,0.755000,3.020000,0.000000
467,2022,MANGUE PIECE,14,0.000,1.484000,3.022000,-0.324000
290,2022,AUBERGINE VIOLETTE,13,6.118,2.860769,1.358462,0.000000
336,2022,BT 3X110MOUCH.CONF,13,0.000,3.475714,3.475714,-0.158214
485,2022,NESCAFE SPEC.FILTR,13,0.000,6.120833,6.120833,-0.050000


,year,name,totalQuantity,totalWeight,unitPrice,price,immediateDiscount
603,2022,TOMATE GRAPPE FRAN,8,10.050,2.365000,2.932500,0.0
290,2022,AUBERGINE VIOLETTE,13,6.118,2.860769,1.358462,0.0
544,2022,POMME GRANNY GROSS,7,5.184,2.442857,1.822857,0.0
512,2022,PDT CONSERVATION,4,4.685,1.690000,2.020000,0.0
548,2022,PORC SAUTE EPAULE,5,4.422,6.120000,4.865000,0.0
546,2022,POMME PINK LADY,4,4.348,3.590000,3.937500,0.0
376,2022,COURGETTE,5,4.223,2.650000,2.176000,0.0
299,2022,BANANE OPEN TOP VR,3,3.539,1.690000,1.993333,0.0
596,2022,TARO VRAC,4,3.513,6.640000,5.902500,0.0
543,2022,POMME GALA MOYENNE,3,3.293,2.390000,2.827500,0.0


# Partial string matching

Thanks to the grouping analysis, we get a grasp of the top products on which we want to make further analysis like actual price evolutions, purchased quantities, variations of item types, etc.

We use regular expressions as well `str.contains()` and fuzzy matching thanks to `rapidfuzz` (for scalability sake).

For each date we have a matching products. In order to perform good matching, we should need to preprocess the labels as to have the dates as tokens that helps to do the matching.

Use `sentence_transformers` for embeddings and compute cosine similarity. For uncertain ones (similarity scores < 0.5), do some normal fuzzy ratio stuff and voting first, second best match and best fuzzy match (based of substring matching rather than contextual matching).

We use Roberta model which is intermediate in terms of performance as the base model is quite lacking on sample analysis.
https://sbert.net/docs/sentence_transformer/pretrained_models.html

In [24]:
def preprocess(text: str): 
    text = text.lower()
    # Normalize to decomposed form (split base characters and diacritics)
    nfkd_form = unicodedata.normalize('NFKD', text)
    text = u"".join([c for c in nfkd_form if not unicodedata.combining(c)])
    
    # Use regex to remove diacritics (Mn = Mark, Nonspacing)
    # text = re.sub(r'\p{Mn}', '', text, flags=re.UNICODE)
    # text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return text.strip()

def fuzzy_match(row, choices, scorer=fuzz.WRatio, processor=None, threshold=80):
    """
    Perform fuzzy matching for a single row against a list of choices based on date criterion.
    Returns the best match and its score if above the threshold.
    """
    result = process.extractOne(row, choices, scorer=scorer, processor=processor, score_cutoff=threshold)
    return result[0] if result is not None else None


In [25]:
def embedding_to_df(
    model,
    df: pd.DataFrame,
    column_name: str,
    new_column_name: str = "embeddingLabel",
    filter_out: str | None = None,
    filter_in: str | None = None
):
    """In-place computation of embedding"""
    if filter_in:
        mask = (df[column_name].notna()) & (df[column_name].str.contains(filter_in))
        df.loc[mask, new_column_name] = df.loc[mask, column_name].apply(
            lambda x: get_cached_encode(model, x)
        )
    else:
        if filter_out:
            mask = (df[column_name].isna()) | (df[column_name].str.contains(filter_out))
        else:
            mask = df[column_name].isna()
        df.loc[~mask, new_column_name] = df.loc[~mask, column_name].apply(
            lambda x: get_cached_encode(model, x)
        )


# Define a cached version of the embedding function
def get_cached_encode(model, text):
    @lru_cache(maxsize=None)  # Cache all results
    def cached_encode(text):
        return model.encode(preprocess(text))
    return cached_encode(text)


def match_labels_df(
    df1: pd.DataFrame, df2: pd.DataFrame, params: Dict[str, Any], criterion: str | None = None
) -> None:
    """Matching names in columns of DataFrames
    Args:
        df1 : with new col
    """
    required_params = ["col1", "col2", "newCol", "groupby1", "groupby2"]
    for param in params.keys():
        if param not in required_params:
            raise KeyError(f"Missing required parameter: {param}")
    # Group rows by date
    grouped_df1 = df1[df1[params["newCol"]].notna()].groupby(params["groupby1"])
    if criterion:
        grouped_df2 = df2[(df2[params["newCol"]].notna()) & (df2[params["col2"]].str.contains(criterion))].groupby(params["groupby2"])
    else:
        grouped_df2 = df2[df2[params["newCol"]].notna()].groupby(params["groupby2"])

    # Iterate over unique dates in df1
    for date, group1 in grouped_df1:
        # group1 = group1.reset_index(drop=True)

        if date in grouped_df2.groups:
            group2 = grouped_df2.get_group(date)
            group2 = group2.reset_index(drop=True)

            # Compute pairwise cosine similarity between embeddings
            similarity_matrix = cosine_similarity(
                np.vstack(group1[params["newCol"]]), np.vstack(group2[params["newCol"]])
            )
            idx = 0
            # Find the best match for each row in group1
            for i, row1 in group1.iterrows():
                # best_match_idx = np.argmax(similarity_matrix[i])

                sort_indices = np.argsort(similarity_matrix[idx])[::-1]
                best_match = group2.iloc[sort_indices[0]]
                similarity_score = similarity_matrix[idx][sort_indices[0]]
                df1.at[i, "matchedLabel1"] = best_match[params["col2"]]
                df1.at[i, "similarity_score1"] = similarity_score

                df1.at[i, "matchedLabelFuzzy"] = fuzzy_match(
                    row1[params["col1"]],
                    choices=group2[params["col2"]].unique(),
                    scorer=fuzz.WRatio,
                    processor=preprocess,
                )

                # Second-best match (if it exists)
                if len(sort_indices) > 1:
                    second_best_match = group2.iloc[sort_indices[1]]
                    df1.at[i, "matchedLabel2"] = second_best_match[params["col2"]]
                    df1.at[i, "similarity_score2"] = similarity_matrix[idx][
                        sort_indices[1]
                    ]
                idx += 1

def last_mode(row):
    modes = row.mode()
    if len(modes) > 1:
        return row.iloc[-1] if row.iloc[-1] else row.iloc[0]
    else:
        return modes.iloc[0]


In [26]:
# Load a pre-trained Sentence Transformer model
# model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer('all-distilroberta-v1') # better model slightly but longer to process

In [27]:
# embedding_to_df(model, df_loyalty, 'itemLabel')
# embedding_to_df(model, df_prod, 'productLabel')
params = {
    "col1" : "itemLabel", 
    "col2" : "productLabel", 
    "newCol" : "embeddingLabel", 
    "groupby1" : "date", 
    "groupby2" : "dateKey"
}
embedding_to_df(
    model,
    df_loyalty,
    params["col1"],
    new_column_name="embeddingLabel",
    filter_out="ART RAYON",
)
embedding_to_df(model, df_prod, params["col2"])

In [28]:
match_labels_df(df_loyalty, df_prod, params)

In [29]:
df_loyalty.loc[df_loyalty.similarity_score1 > 0.5, "matchedLabel"] = df_loyalty.loc[
    df_loyalty.similarity_score1 > 0.5, "matchedLabel1"
]
df_loyalty.loc[df_loyalty.similarity_score1 <= 0.5, "matchedLabel"] = df_loyalty.loc[
    df_loyalty.similarity_score1 <= 0.5,
    ["matchedLabel1", "matchedLabel2", "matchedLabelFuzzy"],
].apply(last_mode, axis=1)

In [31]:
df_loyalty

,operationId,date,earned,burned,itemLabel,promotionLabel,itemRd,loyaltyOperation,embeddingLabel,matchedLabel1,similarity_score1,matchedLabelFuzzy,matchedLabel2,similarity_score2,matchedLabel
0,64070450536,2024-09-28,0.66,0.00,20 OEUFS DJP POULE AU SOL CRF,NaN,0.46,Paiement en caisse,"[0.008895143, 0.002929899, 0.030979047, -0.053...",20 OEUFS DJP POULE,0.857691,20 OEUFS DJP POULE,350G OIGN SAUCIER,0.437943,20 OEUFS DJP POULE
1,64070450536,2024-09-28,0.66,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.20,Paiement en caisse,"[-0.0008011513, 0.018176839, 0.0037206716, 0.0...",BANANE CRF BIO,0.745447,BANANE CRF BIO,PCE BROCOLI FQC F,0.374859,BANANE CRF BIO
2,64063052302,2024-09-24,0.30,0.00,1KG COUSCOUS MOYEN CRF,NaN,0.30,Paiement en caisse,"[-0.0013854598, -0.046837345, 0.00067186187, -...",1KG COUSCOUS MOYEN,0.921145,1KG COUSCOUS MOYEN,190G PESTO ROQUE-B,0.492827,1KG COUSCOUS MOYEN
3,64072667140,2024-09-20,0.20,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.20,Paiement en ligne,"[-0.0008011513, 0.018176839, 0.0037206716, 0.0...",Bananes Bio CARREFOUR BIO,0.678154,Yaourt aux fruits jaunes pêche abricot poire a...,Yaourt aux fruits jaunes pêche abricot poire a...,0.477828,Bananes Bio CARREFOUR BIO
4,64073135027,2024-09-13,0.00,-22.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,65151220627,2025-03-08,21.30,0.00,"ART RAYON FRUITS LEG TVA 5,5",NaN,1.29,Paiement en caisse,NaN,NaN,NaN,NaN,NaN,NaN,NaN
330,65151220627,2025-03-08,21.30,0.00,"ART RAYON POISSONNERIE TVA 5,5",NaN,12.91,Paiement en caisse,NaN,NaN,NaN,NaN,NaN,NaN,NaN
331,65151220627,2025-03-08,21.30,0.00,CHOU FLEUR PIECE,NaN,0.26,Paiement en caisse,"[-0.0059086564, -0.05829449, 0.0066939592, -0....",CHOU FLEUR PIECE,1.000000,CHOU FLEUR PIECE,1L HUILE FLEUR DE,0.558918,CHOU FLEUR PIECE
332,65151220627,2025-03-08,21.30,0.00,BANANE CRF BIO MH 5 FRUITS,NaN,0.25,Paiement en caisse,"[-0.0008011513, 0.018176839, 0.0037206716, 0.0...",BANANE CRF BIO,0.745447,BANANE CRF BIO,125GX8 PAN0% JNES,0.318713,BANANE CRF BIO


In [32]:
df_loyalty.to_csv('data/20250501-carrefour_loyalty_extended.csv', index=False)

In [ ]:
# embedding_to_df(
#     model,
#     df_loyalty,
#     "itemLabel",
#     new_column_name="embeddingLabel",
#     filter_out=None,
#     filter_in="ART RAYON",
# )

In [ ]:
# params_other = {
#     "col1" : "productLabel", 
#     "col2" : "itemLabel", 
#     "newCol" : "embeddingLabel", 
#     "groupby1" : "dateKey", 
#     "groupby2" : "date"
# }

# match_labels_df(df_prod, df_loyalty, params_other, criterion='ART RAYON')

In [46]:
df_loyalty.query("(itemLabel.notna()) & (itemLabel.str.contains('ART RAYON'))").value_counts(["itemLabel"])

itemLabel                     
ART RAYON FRUITS LEG TVA 5,5      23
ART RAYON POISSONNERIE TVA 5,5     8
ART RAYON BOUCHERIE TVA 5,5        4
Name: count, dtype: int64

In [63]:
df_prod.query("(matchedLabel1.notna()) & (matchedLabel1.str.contains('ART RAYON')) & (category == 'food')")[['productLabel', 'category', 'subCategory']].value_counts(["productLabel", "subCategory"])

productLabel                                    subCategory                      
Blanc de dinde fumé FLEURY MICHON               Charcuterie                          1
Chips au Comté BRET'S                           Apéritifs et Chips                   1
Chips saveur cheeseburger LAY'S                 Apéritifs et Chips                   1
Coulommiers Savoureux Et Crémeux PRESIDENT      Fromages                             1
Eau de source CRISTALINE                        Eaux                                 1
Huile de pépins de raisin CARREFOUR CLASSIC'    Huiles, Vinaigres et Vinaigrettes    1
Maïs sans sucres ajoutés CARREFOUR CLASSIC'     Repas de Pâques                      1
Nectar fruit de la passion CARREFOUR SELECTION  Jus de fruits et légumes             1
Name: count, dtype: int64

Given the results, let us determine appropriate thresholds for which we get the matched labels are retained. For matched label with low similarity score, let us say the bottom 10%, we may want to have a vote with the second match and fuzzy.

Naturally, we want a threshold at 0.5, but it is not based on any kind of statistical property. It is kind of practical.

# Categorization

We want to know how many tomatoes we consume or how many detergent we use by year for example. This requires categorization.


In [ ]:
categories = {
    'Tomate' : ['Tomate Cerise', 'Tomate Grappe'],
    'Lessive' : ['ARIEL', 'XTRA'],
    'Nettoyants vaisselle' : ['PAIC', 'MIR'],
    'Papier toilette' : ['Papier toilette'], 
    'Mouchoirs' : ['Mouchoirs'], 
    'Légumes verts' : ['Céleri', 'Poireau', 'BATAVIA'],
    'Légumes non verts' : ['Courgette', 'Aubergine', 'Poivrons', 'Piment', 'Concombre']
}

In [ ]:
df_prod.query("(matchedLabel1.notna()) & (matchedLabel1.str.contains('ART RAYON')) & (category == 'food') & (recordType == 'receipt')")[['productLabel', 'category', 'subCategory']].drop_duplicates('productLabel').to_csv(path_or_buf='data/20250502-carrefour_food_products.csv', index=False)

In [6]:
df_prod.query("((category == 'food') & (recordType == 'order'))")[['productLabel', 'category', 'subCategory']].drop_duplicates('productLabel').to_csv(path_or_buf='data/20250502-carrefour_food_order_products.csv', index=False)

In [ ]:
df_prod.query(
    "productLabel.str.contains('tomate', case=False) & totalWeight > 0"
).value_counts(["recordType", "productLabel", "unitPrice"])